# Stage 13 — Productization

Package the TSM direction model for reuse and handoff:

1. **Refactor** the inline modeling code into reusable functions in `src/prediction.py`
   and confirm they reproduce the notebook results exactly.
2. **Persist** the trained model to `model/model.pkl` and load it (not retrain) on the
   next run.
3. **Serve** it behind a Flask API (`app.py`) and call it live with `requests`.

In [1]:
# --- run me first (notebooks/ -> project/ root) ---
import os
from pathlib import Path
if Path.cwd().name == 'notebooks':
    os.chdir('..')

import numpy as np
import pandas as pd
import requests
import subprocess, sys, time, socket

from src.modeling import load_modeling_data, prepare_features, time_aware_split, run_logistic_classification
from src.prediction import (train_classification_model, predict_direction,
                            load_or_train_model, save_model, latest_feature_row, MODEL_PATH)
%matplotlib inline
print('ok')

ok


## 1. Refactor check

`src.prediction.train_classification_model` + `predict_direction` must reproduce the
notebook's inline logistic result *exactly* — this is how we know the refactor did not
change anything before the code goes into the app.

In [2]:
df = load_modeling_data()
X, _, y_clf = prepare_features(df)
Xtr, Xte, yct, yce = time_aware_split(X, y_clf)

# inline version (Stages 10b / 11)
inline = run_logistic_classification(Xtr, yct, Xte, yce)

# refactored version (src/prediction.py)
refactored = train_classification_model(Xtr, yct)
ref_prob = predict_direction(refactored, Xte[0])['probability_up']

inline_prob = inline['y_prob'][0]
print(f"inline    P(up) on test row 0 : {inline_prob:.8f}")
print(f"refactored P(up) on test row 0 : {ref_prob:.8f}")
assert np.isclose(inline_prob, ref_prob, atol=1e-9), "refactor changed the result"
print("refactor check PASSED: identical predictions")

inline    P(up) on test row 0 : 0.48413191
refactored P(up) on test row 0 : 0.48413191
refactor check PASSED: identical predictions


## 2. Persist the model

Save the production model (trained on the **full** dataset so the deployed API uses all
available history) and confirm it can be loaded back and used.

In [3]:
# Train on the full dataset and save (overwrites any existing model.pkl).
full_model = train_classification_model(X.values, y_clf.values)
save_model(full_model)
print('saved ->', MODEL_PATH)

# Load it back (this is what app.py does at startup) and predict.
reloaded = load_or_train_model()
example = latest_feature_row(df)
print('example (latest day) features:')
print({k: round(v, 4) for k, v in example.items()})
print('prediction:', predict_direction(reloaded, example))

saved -> /Users/dorislee/bootcamp_peiyun_lee/project/model/model.pkl
example (latest day) features:
{'return_1d': -0.0063, 'return_5d': 0.0414, 'return_20d': 0.0323, 'close_to_ma_5': 0.0079, 'close_to_ma_20': 0.0213, 'ma_5_20_spread': 0.0132, 'volume_ratio_20': 0.5878, 'intraday_range': 0.0194, 'overnight_gap': -0.0035}
prediction: {'probability_up': 0.5011938753605787, 'direction': 1, 'direction_label': 'up'}


## 3. Serve it and call the API

Launch `app.py` on port 5001 (5000 is taken by macOS AirPlay Receiver) and exercise the
three routes: health, predict (good + bad input), and run_full_analysis.

In [4]:
def _port_open(port=5001):
    s = socket.socket(); s.settimeout(0.25)
    try:
        s.connect(('127.0.0.1', port)); return True
    except OSError:
        return False
    finally:
        s.close()

if not _port_open():
    subprocess.Popen([sys.executable, 'app.py'],
                     stdout=open('flask_server.log', 'w'), stderr=subprocess.STDOUT,
                     start_new_session=True)
    for _ in range(40):
        if _port_open(): break
        time.sleep(0.25)
print('server up' if _port_open() else 'server NOT up')

server up


In [5]:
BASE = 'http://127.0.0.1:5001'

# health
r0 = requests.get(BASE + '/health', timeout=5)
print('GET  /health            ', r0.status_code, r0.text.strip())

# predict with the latest real feature row
r1 = requests.post(BASE + '/predict', json={'features': example}, timeout=5)
print('POST /predict (good)    ', r1.status_code, r1.text.strip())

# predict with a deliberately bad input -> 400 + JSON error
r2 = requests.post(BASE + '/predict', json={'features': {'return_1d': 0.01}}, timeout=5)
print('POST /predict (bad)     ', r2.status_code, r2.text.strip())

# full analysis
r3 = requests.post(BASE + '/run_full_analysis', timeout=30)
print('POST /run_full_analysis ', r3.status_code)
if r3.status_code == 200:
    print(r3.json())

GET  /health             200 {"n_features":9,"status":"ok"}
POST /predict (good)     200 {"direction":1,"direction_label":"up","probability_up":0.5011938753605787}
POST /predict (bad)      400 {"error":"missing features: ['return_5d', 'return_20d', 'close_to_ma_5', 'close_to_ma_20', 'ma_5_20_spread', 'volume_ratio_20', 'intraday_range', 'overnight_gap']"}
POST /run_full_analysis  200
{'classification': {'accuracy': 0.5, 'auc': 0.5301101480678946, 'precision': 0.5220125786163522, 'recall': 0.532051282051282}, 'n_features': 9, 'regression': {'mae': 0.018833175014646108, 'r2': -0.0005066954407153546, 'rmse': 0.02529181303952601}}


## 4. Notes

- The model is **loaded once at startup** (not inside a route), so the API is fast and
  reproducible.
- **Error handling** returns a JSON error and HTTP 400 for malformed input — never a
  server traceback.
- `requirements.txt` lists `flask` and `joblib` (plus the existing data-science stack);
  `README.md` documents setup, the example requests above, and assumptions/risks.